# 🏥 스마트병동 환자안전 모니터링 — 로컬 실행 버전

Fine-tuned MobileVLM V2 (Base + LoRA Adapter)로 영상을 2초 간격 분석하여 환자 행동 변화 타임라인 보고서를 생성합니다.

> ⚠️ **로컬 환경 주의**: CUDA 없음 → CPU/fp32 추론 → 이미지당 1~2분 예상  
> 향후 OpenVINO INT4 변환 후 1~2초/장으로 개선 예정 (Phase 3)

## 📦 Cell 1: 패키지 설치 (최초 1회)

In [1]:
# ds_study 환경에서 실행: conda activate ds_study
import subprocess
subprocess.run(['pip', 'install', '-q', 'transformers', 'peft', 'pillow', 'opencv-python'], check=True)
print('✅ 패키지 설치 완료')

✅ 패키지 설치 완료


## ⚙️ Cell 2: 경로 & 설정

In [2]:
import os, sys, zipfile, time, cv2
from datetime import datetime
from pathlib import Path
from PIL import Image

# ── 경로 설정 (필요시 수정) ────────────────────────────────────
BASE_DIR      = Path('.')   # 노트북 위치 = 프로젝트 루트
ADAPTER_ZIP   = BASE_DIR / 'mobilevlm_lora_adapter.zip'
ADAPTER_DIR   = BASE_DIR / 'mobilevlm_lora_adapter'
VIDEO_DIR     = BASE_DIR / 'data' / 'demo_videos'
RESULT_DIR    = BASE_DIR / 'data' / 'demo_results'
MOBILEVLM_DIR = BASE_DIR / 'MobileVLM'

# ── 추론 설정 ──────────────────────────────────────────────────
INTERVAL_SEC  = 2                                         # 추론 간격 (초)
PROMPT        = '의료진에게 환자의 현재 상태를 보고하세요.'  # 학습 Q4와 동일
ALERT_KEYWORD = '낙상'                                    # 낙상 감지 키워드
MODEL_NAME    = 'mtgv/MobileVLM_V2-1.7B'

RESULT_DIR.mkdir(exist_ok=True)
print(f'✅ 설정 완료')
print(f'   영상 폴더: {VIDEO_DIR.resolve()}')
print(f'   결과 폴더: {RESULT_DIR.resolve()}')

✅ 설정 완료
   영상 폴더: C:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM\demo_videos
   결과 폴더: C:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM\demo_results


## 🔧 Cell 3: 환경 준비 (레포 클론 & 어댑터 압축 해제)

In [3]:
# MobileVLM 레포 클론 (없을 경우)
if not MOBILEVLM_DIR.exists():
    print('[setup] MobileVLM 레포 클론 중...')
    os.system(f'git clone -q https://github.com/Meituan-AutoML/MobileVLM {MOBILEVLM_DIR}')
    print('[setup] 완료')
sys.path.insert(0, str(MOBILEVLM_DIR))

# LoRA 어댑터 압축 해제 (없을 경우)
if not ADAPTER_DIR.exists():
    if ADAPTER_ZIP.exists():
        print('[setup] LoRA 어댑터 압축 해제 중...')
        with zipfile.ZipFile(ADAPTER_ZIP, 'r') as z:
            z.extractall(BASE_DIR)
        print(f'[setup] ✅ 완료: {ADAPTER_DIR}')
    else:
        print(f'[ERROR] 어댑터 파일이 없습니다: {ADAPTER_ZIP}')
else:
    print(f'✅ 어댑터 이미 존재: {ADAPTER_DIR}')

# 영상 목록 확인
VIDEO_DIR.mkdir(exist_ok=True)
video_files = sorted([v for v in VIDEO_DIR.iterdir() if v.suffix.lower() in ('.mp4', '.avi', '.mov', '.mkv')])
print(f'\n📹 발견된 영상: {len(video_files)}개')
for v in video_files:
    print(f'  - {v.name}')

[setup] LoRA 어댑터 압축 해제 중...
[setup] ✅ 완료: mobilevlm_lora_adapter

📹 발견된 영상: 3개
  - 00105_H_A_SY_C3.mp4
  - 00602_H_D_SY_C3.mp4
  - 00711_H_D_SY_C3.mp4


In [8]:
# 어댑터 폴더 상태 확인
print(f'ADAPTER_DIR 절대경로: {ADAPTER_DIR.resolve()}')
print(f'존재 여부: {ADAPTER_DIR.exists()}')

if ADAPTER_DIR.exists():
    print('폴더 내 파일:')
    for f in ADAPTER_DIR.iterdir():
        print(f'  {f.name}')
else:
    print('❌ 폴더 없음 → zip 압축 해제 시도')
    if ADAPTER_ZIP.exists():
        with zipfile.ZipFile(ADAPTER_ZIP, 'r') as z:
            z.extractall(BASE_DIR)
        print('✅ 압축 해제 완료')
    else:
        print(f'❌ zip 파일도 없음: {ADAPTER_ZIP.resolve()}')

ADAPTER_DIR 절대경로: mobilevlm_lora_adapter
존재 여부: False
❌ 폴더 없음 → zip 압축 해제 시도
✅ 압축 해제 완료


## 🤖 Cell 4: Fine-tuned 모델 로드 (Base + LoRA Adapter)

In [6]:
# MobileVLM 로컬 실행을 위한 전체 패키지 한 번에 설치
import subprocess

packages = [
    'transformers',
    'peft',
    'sentencepiece',   # LlamaTokenizer 필수
    'timm',            # MobileVLM vision encoder 필수
    'torchvision',     # vision ops 필수
    'pillow',
    'opencv-python',
    'einops',          # attention 연산
    'accelerate',      # 모델 로드 최적화
    'safetensors'
]

for pkg in packages:
    result = subprocess.run(['pip', 'install', '-q', pkg], capture_output=True, text=True)
    status = '✅' if result.returncode == 0 else '❌'
    print(f'{status} {pkg}')

print('\n⚠️ 설치 완료 → 반드시 커널 재시작 후 Cell 3부터 다시 실행!')

✅ transformers
✅ peft
✅ sentencepiece
✅ timm
✅ torchvision
✅ pillow
✅ opencv-python
✅ einops
✅ accelerate
✅ safetensors

⚠️ 설치 완료 → 반드시 커널 재시작 후 Cell 3부터 다시 실행!


In [13]:
import zipfile
from pathlib import Path

BASE_DIR  = Path(r'c:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM')
ZIP_PATH  = BASE_DIR / 'mobilevlm_lora_adapter.zip'

print(f'압축 해제 중: {ZIP_PATH}')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    print('zip 내부 파일 목록:')
    for name in z.namelist():
        print(f'  {name}')
    z.extractall(BASE_DIR)

print('\n압축 해제 완료! 폴더 확인:')
adapter_dir = BASE_DIR / 'mobilevlm_lora_adapter'
if adapter_dir.exists():
    for f in adapter_dir.iterdir():
        print(f'  ✅ {f.name}')
else:
    print('❌ 폴더가 생성되지 않았습니다. zip 내부 구조를 확인하세요.')

압축 해제 중: c:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM\mobilevlm_lora_adapter.zip
zip 내부 파일 목록:
  special_tokens_map.json
  tokenizer.model
  tokenizer_config.json
  README.md
  adapter_config.json
  adapter_model.safetensors

압축 해제 완료! 폴더 확인:
❌ 폴더가 생성되지 않았습니다. zip 내부 구조를 확인하세요.


In [14]:
import zipfile, shutil
from pathlib import Path

BASE_DIR    = Path(r'c:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM')
ZIP_PATH    = BASE_DIR / 'mobilevlm_lora_adapter.zip'
ADAPTER_DIR = BASE_DIR / 'mobilevlm_lora_adapter'

# 폴더 생성 후 압축 해제
ADAPTER_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(ADAPTER_DIR)   # ← BASE_DIR 아닌 ADAPTER_DIR에 추출

print('✅ 압축 해제 완료!')
for f in ADAPTER_DIR.iterdir():
    print(f'  {f.name}')


✅ 압축 해제 완료!
  adapter_config.json
  adapter_model.safetensors
  README.md
  special_tokens_map.json
  tokenizer.model
  tokenizer_config.json


In [15]:
import torch
from peft import PeftModel
from mobilevlm.model.mobilevlm import load_pretrained_model
from mobilevlm.utils import disable_torch_init

print('[모델] Base 모델 로드 중... (최초 실행 시 HuggingFace 다운로드 ~3GB)')
print('[모델] CPU/fp32 모드 — CUDA 없음, load_4bit=False (transformers 충돌 회피)')
disable_torch_init()

# 확인된 실제 시그니처: (model_path, load_8bit, load_4bit, device_map, device)
tokenizer, model_base, image_processor, _ = load_pretrained_model(
    MODEL_NAME,
    load_8bit=False,
    load_4bit=False,
    device_map='cpu',   # ← 'auto' 대신 'cpu' 명시 (디스크 오프로드 방지)
    device='cpu',
)

print('[모델] LoRA 어댑터 적용 중...')
model = PeftModel.from_pretrained(model_base, str(ADAPTER_DIR.resolve()))
model.eval()

print(f'✅ Fine-tuned 모델 로드 완료')
print(f'   model:           {type(model)}')
print(f'   tokenizer:       {type(tokenizer)}')
print(f'   image_processor: {type(image_processor)}')

[모델] Base 모델 로드 중... (최초 실행 시 HuggingFace 다운로드 ~3GB)
[모델] CPU/fp32 모드 — CUDA 없음, load_4bit=False (transformers 충돌 회피)
[모델] LoRA 어댑터 적용 중...
✅ Fine-tuned 모델 로드 완료
   model:           <class 'peft.peft_model.PeftModelForCausalLM'>
   tokenizer:       <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>
   image_processor: <class 'transformers.models.clip.image_processing_clip.CLIPImageProcessor'>


## ⚙️ Cell 5: 추론 함수 정의

In [16]:
from mobilevlm.utils import process_images, tokenizer_image_token
from mobilevlm.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from mobilevlm.conversation import conv_templates

def infer_frame(pil_image):
    """PIL 이미지 1장 → 추론 결과 문자열"""
    image_tensor = process_images([pil_image], image_processor, model.config)[0]
    image_tensor = image_tensor.to('cpu', dtype=torch.float32)

    full_prompt = DEFAULT_IMAGE_TOKEN + '\n' + PROMPT
    conv = conv_templates['v1'].copy()   # MobileVLM V2 = Vicuna 기반
    conv.append_message(conv.roles[0], full_prompt)
    conv.append_message(conv.roles[1], None)

    input_ids = tokenizer_image_token(
        conv.get_prompt(), tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt'
    ).unsqueeze(0)

    with torch.inference_mode():
        output_ids = model.generate(
            input_ids,
            images=image_tensor.unsqueeze(0),
            do_sample=False,
            num_beams=3,              # 품질 향상
            min_new_tokens=15,        # 너무 짧은 출력 방지
            max_new_tokens=128,       # 잘림 방지
            repetition_penalty=1.3,   # 반복 억제
            use_cache=True,
        )
    return tokenizer.decode(
        output_ids[0, input_ids.shape[1]:], skip_special_tokens=True
    ).strip()

def format_time(seconds):
    return f'{int(seconds)//60:02d}:{int(seconds)%60:02d}'

print('✅ 추론 함수 정의 완료')
print(f'   프롬프트: {PROMPT}')
print(f'   추론 간격: {INTERVAL_SEC}초 | 경보 키워드: {ALERT_KEYWORD}')

✅ 추론 함수 정의 완료
   프롬프트: 의료진에게 환자의 현재 상태를 보고하세요.
   추론 간격: 2초 | 경보 키워드: 낙상


## 🎬 Cell 6: 영상 선택 & 실시간 추론 실행

In [17]:
# ── 처리할 영상 선택 ───────────────────────────────────────────
# 특정 영상 지정 시: video_path = Path('demo_videos/video1.mp4')
# 전체 처리 시: video_files 리스트 사용

if not video_files:
    print('[ERROR] demo_videos/ 폴더에 영상을 넣어주세요.')
else:
    VIDEO_INDEX = 0   # ← 처리할 영상 번호 (0, 1, 2 ...)
    video_path  = video_files[VIDEO_INDEX]

    cap        = cv2.VideoCapture(str(video_path))
    fps        = cap.get(cv2.CAP_PROP_FPS) or 30
    total_f    = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration   = total_f / fps
    frame_step = max(1, int(fps * INTERVAL_SEC))

    print(f'\n{"="*60}')
    print(f'📹 영상: {video_path.name}')
    print(f'   길이: {format_time(duration)} | FPS: {fps:.1f} | 분석 간격: {INTERVAL_SEC}초')
    print(f'   분석 프레임 수: {total_f // frame_step}장')
    print(f'{"="*60}')
    print('📋 환자 행동 보고서')
    print(f'{"="*60}')

    report      = []
    prev_status = None
    frame_idx   = 0
    alert_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_step != 0:
            frame_idx += 1
            continue

        timestamp  = frame_idx / fps
        time_label = format_time(timestamp)
        pil_img    = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        t0      = time.time()
        status  = infer_frame(pil_img)
        elapsed = time.time() - t0

        if status != prev_status or prev_status is None:
            has_alert = ALERT_KEYWORD in status
            if has_alert:
                alert_count += 1
            alert_str = '  ⚠️ 간호사 호출!' if has_alert else ''
            print(f'{time_label}  {status}{alert_str}  [{elapsed:.1f}s]')
            report.append({'time': time_label, 'status': status, 'alert': has_alert})
            prev_status = status
        else:
            print(f'{time_label}  (상태 유지) [{elapsed:.1f}s]')

        frame_idx += 1

    cap.release()
    print(f'\n✅ 완료 | 상태 변화: {len(report)}회 | 낙상 경보: {alert_count}회')


📹 영상: 00105_H_A_SY_C3.mp4
   길이: 00:10 | FPS: 10.0 | 분석 간격: 2초
   분석 프레임 수: 5장
📋 환자 행동 보고서


RuntimeError: mat1 and mat2 must have the same dtype, but got Float and Half

## 💾 Cell 7: 보고서 저장

In [ ]:
out_path = RESULT_DIR / f'{video_path.stem}_report.txt'

with open(out_path, 'w', encoding='utf-8') as f:
    f.write('환자 행동 모니터링 보고서\n')
    f.write(f'생성일시: {datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
    f.write(f'모델: Fine-tuned MobileVLM V2 1.7B (QLoRA)\n')
    f.write(f'영상: {video_path.name}\n')
    f.write('=' * 40 + '\n')
    for entry in report:
        alert_str = '  ⚠️ 낙상 경보!' if entry['alert'] else ''
        f.write(f"{entry['time']}  {entry['status']}{alert_str}\n")
    f.write('=' * 40 + '\n')
    f.write(f'총 상태 변화: {len(report)}회\n')
    f.write(f'낙상 경보 발생: {alert_count}회\n')

print(f'📄 보고서 저장 완료: {out_path.resolve()}')

print('\n📋 최종 타임라인:')
print('=' * 40)
for entry in report:
    alert_str = '  ⚠️' if entry['alert'] else ''
    print(f"{entry['time']}  {entry['status']}{alert_str}")
print('=' * 40)

## 🔁 Cell 8: 전체 영상 일괄 처리 (선택)
demo_videos/ 안의 모든 영상을 순서대로 처리합니다.

In [ ]:
for vid_path in video_files:
    cap        = cv2.VideoCapture(str(vid_path))
    fps        = cap.get(cv2.CAP_PROP_FPS) or 30
    frame_step = max(1, int(fps * INTERVAL_SEC))

    print(f'\n{"="*60}\n📹 {vid_path.name}\n{"="*60}')

    report_b, prev_status, frame_idx, alert_count = [], None, 0, 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if frame_idx % frame_step != 0:
            frame_idx += 1; continue

        pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        status  = infer_frame(pil_img)
        time_label = format_time(frame_idx / fps)

        if status != prev_status or prev_status is None:
            has_alert = ALERT_KEYWORD in status
            if has_alert: alert_count += 1
            print(f"{time_label}  {status}{'  ⚠️' if has_alert else ''}")
            report_b.append({'time': time_label, 'status': status, 'alert': has_alert})
            prev_status = status
        frame_idx += 1
    cap.release()

    # 보고서 저장
    rpath = RESULT_DIR / f'{vid_path.stem}_report.txt'
    with open(rpath, 'w', encoding='utf-8') as f:
        f.write(f'환자 행동 모니터링 보고서\n생성일시: {datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
        f.write(f'영상: {vid_path.name}\n' + '='*40 + '\n')
        for e in report_b:
            f.write(f"{e['time']}  {e['status']}{'  ⚠️' if e['alert'] else ''}\n")
        f.write(f'총 변화: {len(report_b)}회 | 낙상 경보: {alert_count}회\n')
    print(f'✅ 저장: {rpath}')

print('\n🎉 전체 처리 완료!')